# 📅 2026-05-18 (월) 개발 노트 : 헤더 이미지 복구 + pgvector 임베딩 통합 + 하이브리드 추천 엔진 완성

## 🎯 오늘의 목표
- [x] 4,190개 게임 헤더 이미지 일괄 채우기 (Phase A: CDN URL 패턴)
- [x] 깨진 이미지 검증 (Phase B: HEAD 요청)
- [x] Steam API로 신규 게임 헤더 복구 (Phase C)
- [x] 비게임 데이터(유틸리티) 정리 (is_active=false)
- [x] pgvector 임베딩(1536차원) DB 적재 + HNSW 인덱스 생성
- [x] 하이브리드 추천 엔진 v3 구축 (49차원 지표 + 1536차원 임베딩)
- [x] Claude Code로 22개 파일 전체 주석 작업 + README 작성

## 🛠 진행 상황 및 핵심 기록

### 1️⃣ 헤더 이미지 3단계 복구 작전

#### Phase A: CDN URL 패턴 일괄 적용 (`fill_header_images.py`)
- Steam CDN 표준 패턴 활용: `https://cdn.cloudflare.steamstatic.com/steam/apps/{app_id}/header.jpg`
- 4,190개 전부 일괄 부여 → DB 업데이트 완료
- 결과: 검증 전엔 100% 채워짐

#### Phase B: 실제 이미지 존재 여부 검증 (`verify_header_images.py`)
- aiohttp 비동기 HEAD 요청으로 4,190개 검증
- **391개 실패 (9.3%)** → 빈 문자열로 마킹
- 100개 테스트에서 14개(14%) 실패 발견 → 전체 돌리면 약 580개 예상 → 실제 391개

#### Phase C: Steam API로 신규 게임 복구 (`fix_failed_headers.py`)
- 실패 게임 대부분 app_id 3,000,000번대 = 최근 출시 게임
- 신규 URL 패턴: `https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/{app_id}/{hash}/header.jpg`
- 해시값이 포함돼서 URL 예측 불가능 → Steam API 직접 호출 필요
- **결과: 391개 중 1개만 복구 성공** (Akamai CDN 한국 IP geo-block)
- 결정: 390개는 빈 문자열 유지 → 프론트에서 `onError` fallback 처리

### 2️⃣ 비게임 데이터 정리
- 발견: 3DMark, VEGAS Pro, Auto Clicker, DisplayFusion 등 유틸리티/소프트웨어가 게임 DB에 섞여있음
- 원인: 원본 4,190개가 "Steam 인기 앱 전체"였음 (게임만이 아니라 유틸리티 포함)
- 해결: 장르 기반 필터로 49개 일괄 비활성화
```sql
UPDATE games SET is_active = false 
WHERE genres ILIKE '%유틸리티%' OR genres ILIKE '%오디오 제작%'
   OR genres ILIKE '%동영상 제작%' OR genres ILIKE '%사진 편집%'
   OR genres ILIKE '%애니메이션과 모델링%' OR genres ILIKE '%게임 개발%'
   OR genres ILIKE '%소프트웨어 교육%';
```
- 결과: 활성 게임 4,141개로 정제

### 3️⃣ pgvector 임베딩 통합 (게임 체인저!)

#### 발견: pkl 파일에 이미 1536차원 임베딩 존재
- `data/hidden_gem_data.pkl` 안에 `embedding` 컬럼 발견
- text-embedding-3-small (1536차원), JSON 문자열로 저장
- 4,190개 전부 임베딩 보유

#### pgvector 활용 (이미 v0.8.2 설치되어 있었음)
```sql
ALTER TABLE game_metrics ADD COLUMN IF NOT EXISTS embedding vector(1536);
```

#### 임베딩 적재 (`scripts/load_embeddings.py`)
- pkl에서 임베딩 JSON 파싱 → DB 적재
- **4,141개 적재 완료** (비활성화 49개 제외)
- HNSW 인덱스 생성 (코사인 유사도 기반)
```sql
CREATE INDEX game_metrics_embedding_hnsw
ON game_metrics USING hnsw (embedding vector_cosine_ops)
WITH (m = 16, ef_construction = 64);
```

### 4️⃣ 하이브리드 추천 엔진 v3 (recommender.py)

#### 새로운 추천 공식
**기존 (v2):**
```
코사인(49D) 50% + 유클리드(49D) 50%
```

**신규 (v3) - 하이브리드:**
```
by-game: 코사인(49D) 35% + 유클리드(49D) 35% + 임베딩 코사인(1536D) 30%
by-preference: 유클리드(49D) 60% + 코사인(49D) 40% (임베딩 미사용)
```

#### 검증 결과 (Undertale 기반 추천)
| 순위 | 게임 | 점수 | 평가 |
|------|------|------|------|
| 1 | DELTARUNE | 0.9309 | 완벽한 1순위 (같은 개발자) |
| 2 | The Stanley Parable | 0.7774 | 메타 서사 결 매칭 |
| 3 | Pikuniku | 0.7764 | 유머/인디 감성 매칭 |

→ 임베딩 추가로 "의미적 유사성" 포착 능력 향상

### 5️⃣ Claude Code로 22개 파일 주석 작업
- 한국어 + 영어 혼용 주석 스타일 도입
- docstring: 첫 줄 EN 요약 + 둘째 줄 KR 상세 설명
- 인라인: `# 한국어 / English` 패턴

#### 작업 완료 파일
**fastapi_app/ (7개)**
- config.py, database.py, main.py, models/game.py, schemas/game.py, routers/games.py, services/recommender.py

**django_core/ (8개)**
- config/urls.py, config/settings.py, apps/users/models.py, apps/users/admin.py, apps/games/admin.py, apps/games/signals.py, management/commands/load_games.py, management/commands/analyze_new_games.py

**scripts/ (14개)**
- 데이터 파이프라인 전체 + 헤더 이미지 + 임베딩 로더 등 모두 docstring 추가

### 6️⃣ Claude.ai Projects Custom Instructions 설정
프로젝트 설정에 코드 작성 규칙 박아둠:
```
## 코드 작성 규칙
- docstring: 첫 줄 EN 요약, 둘째 줄 KR 상세 설명
- 인라인 주석: # 한국어 설명 / English description 혼용
- 함수/클래스 전부 docstring 필수
- 풀코드로 줄 것 (부분 수정 X)
- Python 3.11+, async/await, type hints 필수
- Pydantic v2 (ConfigDict), SQLAlchemy 2.0 비동기
```

### 📌 기억해야 할 정보
- **pgvector 버전**: 0.8.2 (HNSW 지원)
- **임베딩 모델**: text-embedding-3-small (1536차원)
- **활성 게임**: 4,141개 (49개 비활성화 후)
- **헤더 이미지 정상**: 3,750개 (90.7%)
- **헤더 이미지 빈값**: 390개 (Akamai geo-block)
- **Claude Code 설치**: `npm install -g @anthropic-ai/claude-code`
- **HNSW 인덱스명**: `game_metrics_embedding_hnsw`

## 🚨 트러블슈팅 (문제 및 해결)

### 문제 1: 최근 출시 게임(app_id 3,000,000+)의 헤더 이미지 404
- **문제:** Phase A로 부여한 CDN URL 중 391개가 404 응답
- **원인:** Steam이 최근 게임은 새 CDN 패턴(`shared.akamai.steamstatic.com/store_item_assets/...`) 사용. 해시값이 URL에 포함되어 예측 불가능
- **해결 시도:** 
  - Steam API(`appdetails`)에서 정확한 URL 가져오기 시도
  - 처음엔 brower로 테스트 성공했으나, 대량 요청 시 차단됨
- **최종 결정:** 390개는 빈 문자열 유지. 프론트에서 `<img onError>` fallback 처리

### 문제 2: Steam API Akamai geo-block
- **문제:** `fix_failed_headers.py`로 Steam API 호출 시 391개 중 1개만 복구 성공
- **에러:** `ConnectionResetError: [WinError 10054]` + Access Denied (Akamai)
- **원인:** 한국 IP + 프로그래밍 방식 대량 요청 → Akamai CDN이 차단
- **시도1:** concurrency 5 → 1로 낮춤 (50it/s까지만 떨어짐)
- **시도2:** User-Agent 추가, 딜레이 추가 → 효과 미미
- **결론:** 한국 IP 차단이라 우회 불가. 프록시/VPN 없이는 해결 어려움. 프론트 fallback으로 우회

### 문제 3: `fix_failed_headers.py`에서 API 두 번 호출 버그
- **문제:** `as_completed`로 한 번, `gather`로 또 한 번 호출 → 세션 꼬여 전부 None 반환
- **해결:** `gather` 단일 호출로 정리. `content_type=None` 추가하여 JSON 파싱 강제

### 문제 4: 비게임 데이터 혼재 (3DMark, VEGAS Pro 등)
- **문제:** gem_potential 낮은 항목 조회 시 게임이 아닌 유틸리티 발견
- **원인:** 원본 4,190개가 "Steam 인기 앱 전체" 였음 (게임 한정 X)
- **해결:** 장르 기반 SQL UPDATE로 49개 일괄 `is_active=false` 처리

### 문제 5: pkl 임베딩이 str 타입으로 저장됨
- **문제:** `data['embedding'][0]` 출력 시 차원이 19232로 표시
- **원인:** 임베딩이 JSON 문자열로 저장돼 있어 `len()`이 문자 개수를 셈
- **해결:** `json.loads()` 파싱 후 확인 → 1536차원 (text-embedding-3-small) 확인

### 문제 6: SQLAlchemy ORM이 embedding 컬럼 인식 안 됨
- **문제:** `update(GameMetric).values(embedding=...)` 실행 시 에러
- **원인:** ORM 모델(`game.py`)에 embedding 컬럼이 정의되지 않음
- **해결:** raw SQL로 우회 (`text("UPDATE game_metrics SET embedding = :emb WHERE game_id = :gid")`)
- **추후 개선:** `from pgvector.sqlalchemy import Vector` 추가 후 `Column(Vector(1536))`로 정식 통합

### 문제 7: pkl 파일 numpy 호환성 에러
- **문제:** `ModuleNotFoundError: No module named 'numpy._core.numeric'`
- **원인:** 저장 시 numpy 버전과 로드 시 numpy 버전 불일치
- **해결:** `pip install --upgrade numpy`

## 💡 인사이트 및 다음 할 일

### 배운 점
1. **CDN URL 패턴의 진화**: 같은 서비스도 시간에 따라 URL 구조 변경. 해시 기반 URL은 예측 불가능 → API 의존 필수
2. **Geo-block의 현실적 영향**: 글로벌 API 사용 시 한국 IP 차단 가능성 항상 염두에 둘 것
3. **pgvector + HNSW의 위력**: 1536차원 벡터에서도 sub-second 검색 가능. 4,141개 정도는 인덱스 없어도 빠름
4. **하이브리드 추천의 강점**: 구조화 지표(49D)는 명시적 매칭, 임베딩(1536D)은 의미적 매칭. 둘 다 결합 시 추천 품질 ↑
5. **데이터 출처 확인의 중요성**: '인디 게임'인 줄 알았는데 'Steam 인기 앱 전체'였음. 데이터 도메인 검증 필수
6. **Claude Code의 효율**: 22개 파일 주석 작업을 한 번에 처리. 토큰 효율을 위해 작업 우선순위 정해서 분할 추천
7. **pgvector 컬럼 추가 시 ORM 동기화**: raw SQL 우회는 임시 방편. 정식으로 `Vector` 타입 import 권장
8. **`onError` fallback 패턴의 가치**: 90.7% 데이터만 있어도 서비스 가능. 100% 추구하다 시간 낭비 X

### 다음 할 일 (우선순위)

#### 🥇 1순위: Next.js 프론트엔드 시작
- **기술 스택 확정**:
  - Next.js 15 (App Router)
  - TypeScript
  - Tailwind CSS v4
  - shadcn/ui
  - TanStack Query v5
  - Zustand (상태관리)
  - Framer Motion (애니메이션)
- **첫 화면 설계**: 랜딩 → 검색/추천 페이지
- **이미지 fallback**: `onError` 핸들러로 placeholder 처리
- **배포**: Vercel (프론트) + Railway/Render (FastAPI)

#### 🥈 2순위: 신작 게임 수집 + Few-shot 파이프라인
- Steam API로 최근 출시 게임 목록 수집
- 4,190개 데이터를 Few-shot 예시로 활용 → `gpt-4.1-mini`로 비용 절감
- text-embedding-3-small로 신규 임베딩 생성
- 자동화: cron/celery로 주기적 실행

#### 🥉 3순위: 유저 시스템
- 소셜 로그인 (OAuth)
- 좋아요/싫어요 → taste_dna 학습
- 추천 히스토리 저장
- 사용자 데이터 분석 → 추천 알고리즘 재학습 데이터로 활용

#### 🏅 4순위: ORM 정식 통합 + 추가 메타데이터
- `models/game.py`에 `Column(Vector(1536))` 정식 추가
- Steam API로 빠진 메타데이터 채우기 (developer, review_count, release_date, price)
- 데이터 품질 ↑ → 추천 정확도 ↑

## 📊 오늘의 통계

| 항목 | 수치 |
|------|------|
| 헤더 이미지 복구 성공 | 3,750개 (90.7%) |
| 헤더 이미지 복구 실패 | 390개 (Akamai geo-block) |
| 비활성화한 비게임 데이터 | 49개 (유틸리티) |
| 최종 활성 게임 | 4,141개 |
| 적재한 임베딩 | 4,141개 × 1536차원 |
| 추가한 인덱스 | 1개 (HNSW, 코사인) |
| 추천 엔진 차원 확장 | 49D → 49D + 1536D 하이브리드 |
| 작성한 신규 스크립트 | 4개 (fill/verify/fix_failed/load_embeddings) |
| 주석 작업 완료 파일 | 22개 (Claude Code 활용) |
| 작성한 README | 3개 (root + scripts/ + 폴더별) |

## 🎯 백엔드 최종 완성도

| 영역 | 상태 |
|------|------|
| 게임 DB | ✅ 4,141개 활성 |
| 60개 지표 | ✅ 99.6% 완벽 |
| 헤더 이미지 | ✅ 90.7% |
| 1536차원 임베딩 | ✅ 100% 적재 |
| HNSW 인덱스 | ✅ 생성 완료 |
| 추천 엔진 | ✅ 하이브리드 v3 |
| API 엔드포인트 | ✅ 6개 검증 완료 |
| 코드 주석 | ✅ 22개 파일 완료 |
| README | ✅ 3개 작성 완료 |

## 🔗 핵심 명령어 치트시트

```bash
# 헤더 이미지 작업
python scripts/fill_header_images.py
python scripts/verify_header_images.py
python scripts/fix_failed_headers.py --concurrency 1

# 비게임 데이터 비활성화
docker exec -it hidden_gem_db psql -U juntae -d hidden_gem_db -c "
UPDATE games SET is_active = false WHERE genres ILIKE '%유틸리티%' OR ...;"

# pgvector 컬럼 추가
docker exec -it hidden_gem_db psql -U juntae -d hidden_gem_db -c "
ALTER TABLE game_metrics ADD COLUMN IF NOT EXISTS embedding vector(1536);"

# 임베딩 적재 + HNSW 인덱스
python scripts/load_embeddings.py

# 하이브리드 추천 테스트
curl -X POST http://localhost:8000/api/v1/games/recommend/by-game \
  -H "Content-Type: application/json" \
  -d '{"app_id": 391540, "count": 5}'  # Undertale

# Claude Code 실행
npm install -g @anthropic-ai/claude-code
claude  # 프로젝트 루트에서
```

## 📁 폴더 구조 (5월 18일 기준 최종)

```
C:\Hidden-Gem-project\
├── README.md                          # ✨ 최신화 완료
├── CLAUDE.md                          # 코드 작성 규칙
├── django_core/                       # ✨ 주석 완료
│   ├── README.md
│   ├── config/
│   └── apps/
│       ├── users/
│       └── games/
├── fastapi_app/                       # ✨ 주석 완료
│   ├── README.md
│   ├── main.py
│   ├── config.py
│   ├── database.py
│   ├── models/game.py                 # 60지표 + (Vector 추가 예정)
│   ├── schemas/game.py
│   ├── services/recommender.py        # 하이브리드 v3 ⭐
│   ├── routers/games.py
│   └── legacy/
├── scripts/                           # ✨ 주석 완료
│   ├── README.md
│   ├── fill_header_images.py          # 신규 ⭐
│   ├── verify_header_images.py        # 신규 ⭐
│   ├── fix_failed_headers.py          # 신규 ⭐
│   ├── load_embeddings.py             # 신규 ⭐
│   ├── fix_data.py
│   ├── extract_sample.py
│   └── (데이터 파이프라인 9개)
└── data/
    ├── hidden_gem_data.csv
    ├── hidden_gem_data.pkl            # 1536D 임베딩 보유
    └── final/
        └── final_master_games_fixed.jsonl
```

## 🚀 내일 시작 지점

**프론트엔드 첫 단계**:
1. `frontend/` 폴더 생성
2. `npx create-next-app@latest` 실행
3. TypeScript + Tailwind + App Router 선택
4. shadcn/ui 초기화
5. TanStack Query 설정
6. API 클라이언트 작성 (`lib/api.ts`)
7. 랜딩 페이지 → 추천 페이지 흐름 설계